# Train FastReID on RandPerson (Colab GPU)

Trains a **bagtricks-R50** person-ReID model on **RandPerson** — a synthetic,
**Apache-2.0, commercial-safe** dataset (no privacy issues). The output weights
(`model_final.pth`) drop straight into your local pipeline: set
`reid.weights: models/fastreid/model_final.pth` in `config.yaml`.

**Filename format** (how labels are parsed): `000000_s00_c00_f000264.jpg`
= personID `000000`, scene `s00`, camera `c00`, frame `f000264`.

### BEFORE YOU RUN
1. Menu: **Runtime -> Change runtime type -> Hardware accelerator = GPU (T4)**.
2. Run the cells top to bottom. Total time ~2-3 hours on a T4 for 30 epochs.
3. Free Colab disconnects if idle — keep the tab open.

## 1. Confirm the GPU is on

In [ ]:
import torch
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU! Runtime -> Change runtime type -> GPU."
!nvidia-smi -L

## 2. Get FastReID + install dependencies

We clone the official FastReID repo. (If you'd rather use your own vendored copy,
upload your `fast-reid/` folder instead of cloning.)

In [ ]:
%cd /content
![ -d fast-reid ] || git clone https://github.com/JDAI-CV/fast-reid.git
%pip -q install gdown yacs termcolor tabulate scikit-learn tensorboard faiss-cpu Cython opencv-python-headless
print("deps installed")

## 3. Download RandPerson (~132k images)

Downloads the image subset from the authors' Google Drive and extracts it to
`/content/datasets/randperson_subset/`. This is a few GB — give it a few minutes.

In [ ]:
import os, zipfile, tarfile, glob
os.makedirs("/content/datasets/randperson_subset", exist_ok=True)

# Google Drive file id from the RandPerson README.
FILE_ID = "1qIFvTPt1q37_xWeGbFumZag6uAx9c4--"
archive = "/content/randperson_archive"
if not glob.glob("/content/datasets/randperson_subset/**/*.jpg", recursive=True):
    !gdown {FILE_ID} -O {archive}
    # The archive may be a .zip or a .tar(.gz) -- handle both.
    try:
        with zipfile.ZipFile(archive) as z:
            z.extractall("/content/datasets/randperson_subset")
        print("extracted zip")
    except zipfile.BadZipFile:
        with tarfile.open(archive) as t:
            t.extractall("/content/datasets/randperson_subset")
        print("extracted tar")
else:
    print("images already present, skipping download")

In [ ]:
# sanity: count images and show example filenames (must match the s##_c## format)
imgs = glob.glob("/content/datasets/randperson_subset/**/*.jpg", recursive=True)
print("total images:", len(imgs))
for p in imgs[:5]:
    print("  ", os.path.basename(p))
assert imgs, "No images found -- check the download/extraction path."
# If the names do NOT look like 000000_s00_c00_f000264.jpg, the regex in the
# next cell needs adjusting to match the actual format.

## 4. Register RandPerson as a FastReID dataset

FastReID doesn't ship a RandPerson loader, so we write one. It parses id + camera
from each filename and — importantly — **holds out 100 identities for evaluation**
(so the Rank-1/mAP you see during training is measured on people the model never
trained on). Train ids/cams are prefixed with the dataset name, exactly like
FastReID's own Market-1501 loader.

In [ ]:
dataset_code = """# encoding: utf-8
import glob
import os.path as osp
import re
from collections import defaultdict

from .bases import ImageDataset
from ..datasets import DATASET_REGISTRY


@DATASET_REGISTRY.register()
class RandPerson(ImageDataset):
    \"\"\"RandPerson synthetic ReID (Apache-2.0). Filenames like
    000000_s00_c00_f000264.jpg -> personID, scene, camera, frame.\"\"\"

    dataset_dir = "randperson_subset"
    dataset_name = "randperson"
    _PAT = re.compile(r"(\\d+)_s(\\d+)_c(\\d+)_f(\\d+)")
    NUM_EVAL_IDS = 100   # identities held out for evaluation

    def __init__(self, root="datasets", **kwargs):
        self.root = root
        self.data_dir = osp.join(self.root, self.dataset_dir)
        self.check_before_run([self.data_dir])

        img_paths = glob.glob(osp.join(self.data_dir, "**", "*.jpg"), recursive=True)
        if not img_paths:
            raise RuntimeError("No .jpg under " + self.data_dir)

        by_pid = defaultdict(lambda: defaultdict(list))   # pid -> camid -> [paths]
        for p in img_paths:
            m = self._PAT.search(osp.basename(p))
            if not m:
                continue
            pid, scene, cam, _ = map(int, m.groups())
            camid = scene * 100 + cam                      # unique camera per scene
            by_pid[pid][camid].append(p)

        multicam = sorted(pid for pid, cams in by_pid.items() if len(cams) >= 2)
        eval_pids = set(multicam[: self.NUM_EVAL_IDS])

        train, query, gallery = [], [], []
        for pid, cams in by_pid.items():
            if pid in eval_pids:
                cam_ids = sorted(cams)
                qcam = cam_ids[0]
                query.append((cams[qcam][0], pid, qcam))          # 1 query image
                for c in cam_ids:                                  # rest -> gallery
                    imgs = cams[c][1:] if c == qcam else cams[c]
                    for ip in imgs:
                        gallery.append((ip, pid, c))
            else:
                for c, imgs in cams.items():
                    for ip in imgs:
                        train.append((ip, f"{self.dataset_name}_{pid}",
                                      f"{self.dataset_name}_{c}"))

        super().__init__(train, query, gallery, **kwargs)
"""

path = "/content/fast-reid/fastreid/data/datasets/randperson.py"
with open(path, "w") as f:
    f.write(dataset_code)

# make sure it gets imported (registered) by the datasets package
init_path = "/content/fast-reid/fastreid/data/datasets/__init__.py"
with open(init_path) as f:
    init = f.read()
if "randperson" not in init:
    with open(init_path, "a") as f:
        f.write("\nfrom .randperson import RandPerson\n")
print("RandPerson dataset registered.")

## 5. Write the training config

Fine-tunes from the ImageNet backbone (Base-bagtricks sets `PRETRAIN: True`).
30 epochs is a good first run; bump `MAX_EPOCH` later for more accuracy.

In [ ]:
cfg_yaml = """_BASE_: ../Base-bagtricks.yml

DATASETS:
  NAMES: ("RandPerson",)
  TESTS: ("RandPerson",)

SOLVER:
  IMS_PER_BATCH: 64
  MAX_EPOCH: 30
  STEPS: [15, 25]
  CHECKPOINT_PERIOD: 10

DATALOADER:
  NUM_WORKERS: 2

TEST:
  EVAL_PERIOD: 10
  IMS_PER_BATCH: 128

OUTPUT_DIR: logs/randperson/bagtricks_R50
"""
import os
os.makedirs("/content/fast-reid/configs/RandPerson", exist_ok=True)
with open("/content/fast-reid/configs/RandPerson/bagtricks_R50.yml", "w") as f:
    f.write(cfg_yaml)
print("config written")

## 5b. Compatibility patch (FastReID + Python 3.12)

Colab runs Python 3.12, but FastReID was written for 3.6-3.7 and uses a few APIs
that later Python/NumPy removed:
- `from collections import Mapping` -> moved to `collections.abc` in 3.10.
- `np.int` / `np.float` -> removed in NumPy 1.24+ (use plain `int`/`float`).

This cell rewrites those across the FastReID source so training can import.
(Idempotent + only touches the affected lines. Training runs in a fresh
subprocess, so no kernel restart is needed after patching.)

In [ ]:
import re, glob

ABC = {"Mapping","MutableMapping","Sequence","MutableSequence","Iterable",
       "Callable","Set","MutableSet","Hashable","Container","Sized","Collection"}

def fix_collections(src):
    def repl(m):
        names = [n.strip() for n in m.group(1).split(",")]
        abc   = [n for n in names if n in ABC]
        plain = [n for n in names if n not in ABC]
        out = []
        if plain: out.append("from collections import " + ", ".join(plain))
        if abc:   out.append("from collections.abc import " + ", ".join(abc))
        return "\n".join(out)
    src = re.sub(r"from collections import ([^\n(]+)", repl, src)
    for name in ABC:                                  # attribute-style usages
        src = re.sub(r"collections\." + name + r"\b", "collections.abc." + name, src)
    return src

NP = {r"\bnp\.int\b": "int", r"\bnp\.float\b": "float", r"\bnp\.bool\b": "bool",
      r"\bnp\.object\b": "object", r"\bnp\.long\b": "int", r"\bnp\.str\b": "str"}

changed = 0
for path in glob.glob("/content/fast-reid/**/*.py", recursive=True):
    s = open(path).read(); orig = s
    s = fix_collections(s)
    for pat, rep in NP.items():
        s = re.sub(pat, rep, s)
    if s != orig:
        open(path, "w").write(s); changed += 1; print("patched", path)
print("files patched:", changed)

## 6. Train

Point FastReID at our dataset folder via `FASTREID_DATASETS`, then launch. You'll
see the loss fall and, every 10 epochs, Rank-1 / mAP on the held-out identities.

In [ ]:
import os
os.environ["FASTREID_DATASETS"] = "/content/datasets"
%cd /content/fast-reid
!python tools/train_net.py --config-file configs/RandPerson/bagtricks_R50.yml --num-gpus 1

## 7. Save the weights

The trained model is at `logs/randperson/bagtricks_R50/model_final.pth`. Save it
to your Google Drive (survives disconnects), and/or download it directly.

In [ ]:
WEIGHTS = "/content/fast-reid/logs/randperson/bagtricks_R50/model_final.pth"
import os; print("exists:", os.path.exists(WEIGHTS),
                 "| size MB:", round(os.path.getsize(WEIGHTS)/1e6, 1) if os.path.exists(WEIGHTS) else 0)

# Option A: copy to Google Drive
from google.colab import drive
drive.mount("/content/drive")
!cp {WEIGHTS} /content/drive/MyDrive/randperson_bagtricks_R50.pth
print("copied to Drive: MyDrive/randperson_bagtricks_R50.pth")

# Option B: direct download
# from google.colab import files; files.download(WEIGHTS)

## 8. Use it locally

1. Copy the `.pth` into your project: `models/fastreid/model_final.pth`.
2. In `config.yaml` set:
   ```yaml
   reid:
     weights: models/fastreid/model_final.pth
   ```
3. Re-run `python check_reid.py` — the same/diff **GAP should widen sharply** and
   the gallery should stop merging everyone into one id. Then `python main.py`.

That's the whole loop: synthetic, commercial-safe data -> real embeddings ->
meaningful global ids across your cameras.